In [1]:
from dataclasses import dataclass
from datetime import date
import os
from pathlib import Path
import tomllib
import pandas as pd
import geopandas as gpd
from sqlalchemy import create_engine
from nvi_etl.geo_reference import (
    pull_city_boundary, 
    pull_council_districts, 
    pull_zones, 
    pull_cdo_boundaries,
)
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows


WORKING_DIR = Path(os.getcwd()).parent.parent


with open(WORKING_DIR / "config.toml", "rb") as f:
    config = tomllib.load(f)


def get_engine(db_name=None):
    user = config["db"]["user"]
    password=config["db"]["password"]
    database=db_name or config["db"]["database"]
    host=config["db"]["host"]

    return create_engine(f"postgresql+psycopg://{user}:{password}@{host}:5432/{database}")

In [17]:
districts = pull_council_districts(2026)
zones = pull_zones(2026)

In [2]:
def combine_survey_and_geocoded(frame, geoframe):
    return gpd.GeoDataFrame(
        frame.rename(
            columns={
                "Response ID": "response_id",
            }
        ).merge(
            geoframe.rename(
                columns={
                    "Response_I": "response_id",
                }
            )[["response_id", "geometry"]], 
            on="response_id"
        )
    )

In [28]:
frame = pd.read_csv(
    "Q:\\3_Projects\\NVI\\2025\\nvi_survey_data_2025_20260226.csv", 
    low_memory=False
).rename(columns={"Response ID": "response_id"})

geoframe = gpd.read_file(
    "Q:\\3_Projects\\NVI\\2025\\Final Shapefiles\\Final2025NVIDataset_cleaned_20260304.shp"
).rename(columns={"Response_I": "response_id"})

datadictionary = pd.read_excel(WORKING_DIR / "primary_survey" / "v2025" / "conf" / "nvi_answer_key_20260316.xlsx")

location_dictionary = pd.read_excel(
    WORKING_DIR / "primary_survey" / "v2025" / "conf" / "locations_20260312.xlsx", dtype={"location": str}
)

geocoded = combine_survey_and_geocoded(frame, geoframe)

# # NOTE SC: Commented out joins for zones and districts for now 
# # NOTE MV: I've uncommented, because we can allow this to flow into the full response, 
# # and then filter later. We have plans to include the CDOs in the database.
geocoded = (
    geocoded
    .to_crs(2898)
    .sjoin(districts[["geometry", "district_number"]], how="left", predicate="within")
    .drop(columns="index_right")
)

geocoded = (
    geocoded
    .to_crs(2898)
    .sjoin(zones[["geometry", "zone_id"]], how="left", predicate="within")
)

In [75]:
# Let's make sense sure we didn't lose any rows
assert len(frame) == len(geocoded)

In [76]:
@dataclass
class Table:
    topic: str | None
    question: str 
    table: pd.DataFrame | None = None
    summary_level: str | None = None


def make_table(column, table=None, summary_level=None):
    return Table(
        column["topic_text"] or None,
        column["question_text"],
        table,
        summary_level=summary_level,
    )

In [87]:
result = {}
serious = []
errors = []
# Loop through each question
for i, column in datadictionary.drop_duplicates(subset="full_column").iterrows():
    if (
        (column["start_date"] > pd.Timestamp(date(year=2026, month=1, day=1)))
        or (column["end_date"] < pd.Timestamp(date(year=2026, month=1, day=1)))
    ):
        continue

    tables = []

    topic_text = column["topic_text"]
    column_name = column["full_column"].replace("'", "’")

    if column_name not in frame:
        print(column["indicator_db_id"])
        errors.append(column_name)
        continue

    labels = datadictionary[
        datadictionary["full_column"] == column_name
    ][["survey_code", "answer"]].set_index("survey_code")

    # First aggregate city-wide
    out = (
        geocoded[column_name]
        .value_counts(dropna=False)
        .to_frame()
    )

    try:
        out.index = out.index.astype(pd.Int64Dtype())
    except ValueError:
        errors.append("citywide")
        continue

    out = (
        out
        .join(labels)
        .sort_values(column_name)
        .reset_index(drop=True)
    )[["answer", "count"]]

    tables.append(make_table(column, out, "citywide"))


    # Then aggregate districts, zones

    for summary in ("district_number", "zone_id"):
        out = (
            geocoded[[column_name, summary]]
            .value_counts(dropna=False)
            .reset_index(summary)
        )

        out.index = out.index.astype(pd.Int64Dtype())

        out = (
            out
            .join(labels)
            .sort_values([summary, column_name])
            .reset_index(drop=True)
        )[[summary, "answer", "count"]]

        tables.append(make_table(column, out, summary))

    result[column["question_text"]] = tables

5.0
54.0
62.0
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan


In [78]:
wb = Workbook()
toc = wb.active
toc.title = "Table of Contents"
toc.append(["Sheet #", "Question"])

for sheet_num, (question, tables) in enumerate(result.items(), 1):
    ws = wb.create_sheet(title=f"Q{sheet_num}")
    toc.append([sheet_num, question])
    first = tables[0]
    if first.topic:
        ws.append([first.topic])
    ws.append([first.question])
    for table in tables:
        if table.summary_level:
            ws.append([table.summary_level])
        for r in dataframe_to_rows(table.table, index=False, header=True):
            ws.append(r)
        ws.append([])

wb.save("survey_output_20260312.xlsx")

In [ ]:
def append_universe_and_percentages(table):
    included = table["universe_include"]

    # Calculate a percentage column
    total = (
        table[included]
        .groupby("location")["count"]
        .transform("sum")
    )

    table["universe"] = total
    table.loc[included, "percentage"] = table.loc[included, "count"] / total

    return table


def aggregate_group():
    pass


def aggregate_question(frame: pd.DataFrame, column_name: str, 
    summary: str, labels: pd.DataFrame) -> pd.DataFrame:


    aggregation_type = labels.iloc[0]["response_type"]
    """
    This assumes question is answered with a selection from a likert item


    in:
        frame: our current dataframe
        column_name: the name of the frame we're aggregating
        summary: the summary level we're working with -- another column
        labels: rows of the datadictionary associated with the column name

    out:
        The rows aggregated
    """
    table = (
        frame[[column_name, summary]]
        .value_counts(dropna=False)
        .reset_index(summary)
    )

    # Reference for filling in join misses
    reference = labels.iloc[0]

    table.index = table.index.astype(pd.Int64Dtype())

    table = (
        table
        .join(labels, how="outer")
        .sort_values([summary, column_name])
        .reset_index(drop=True)
        .rename(columns={summary: "location"})
        .fillna({
            "topic_text": reference["topic_text"],
            "question_text": reference["question_text"],
            "full_column": reference["full_column"],
            "answer": "[SKIPPED]",
            "db_question_code": reference["db_question_code"],
            "indicator_include": False,
            "universe_include": False,
            "indicator_db_id": reference["indicator_db_id"],
            "location": "[NO ADDRESS PROVIDED]",
        })
        .astype({
            "db_question_code": pd.Int64Dtype(), 
            "db_answer_code": pd.Int64Dtype(),
            "indicator_db_id": pd.Int64Dtype(),
            "universe_include": bool,
            "indicator_include": bool,
        })
        .assign(summary_level=summary)
    )

    return append_universe_and_percentages(table)

In [ ]:
# Try likert -- checks out

column_name = "Stay_12_Months:Agreement_Statements_Access_and_Support"
labels = datadictionary[datadictionary["full_column"] == column_name]
summary = "district_number"

aggregate_question(geocoded, column_name, summary, labels)

,location,count,indicator_db_id,indicator_id,indicator_include,universe_include,group,topic_text,question,question_text,...,full_column,response_type,universe_query,site_category,and_or,start_date,end_date,summary_level,universe,percentage
0,1,13,1,NaN,True,True,Agreement_Statements_Access_and_Support,Please indicate how much you agree to the foll...,Stay_12_Months,I plan to stay in my neighborhood for the next...,...,Stay_12_Months:Agreement_Statements_Access_and...,GROUPED-SINGLE,@ALL,Community Capacity,OR,2023-01-01,2099-12-31,district_number,1006.0,0.012922
1,1,51,1,NaN,True,True,Agreement_Statements_Access_and_Support,Please indicate how much you agree to the foll...,Stay_12_Months,I plan to stay in my neighborhood for the next...,...,Stay_12_Months:Agreement_Statements_Access_and...,GROUPED-SINGLE,@ALL,Community Capacity,OR,2023-01-01,2099-12-31,district_number,1006.0,0.050696
2,1,34,1,NaN,False,True,Agreement_Statements_Access_and_Support,Please indicate how much you agree to the foll...,Stay_12_Months,I plan to stay in my neighborhood for the next...,...,Stay_12_Months:Agreement_Statements_Access_and...,GROUPED-SINGLE,@ALL,Community Capacity,OR,2023-01-01,2099-12-31,district_number,1006.0,0.033797
3,1,75,1,NaN,False,True,Agreement_Statements_Access_and_Support,Please indicate how much you agree to the foll...,Stay_12_Months,I plan to stay in my neighborhood for the next...,...,Stay_12_Months:Agreement_Statements_Access_and...,GROUPED-SINGLE,@ALL,Community Capacity,OR,2023-01-01,2099-12-31,district_number,1006.0,0.074553
4,1,170,1,NaN,False,True,Agreement_Statements_Access_and_Support,Please indicate how much you agree to the foll...,Stay_12_Months,I plan to stay in my neighborhood for the next...,...,Stay_12_Months:Agreement_Statements_Access_and...,GROUPED-SINGLE,@ALL,Community Capacity,OR,2023-01-01,2099-12-31,district_number,1006.0,0.168986
5,1,663,1,NaN,False,True,Agreement_Statements_Access_and_Support,Please indicate how much you agree to the foll...,Stay_12_Months,I plan to stay in my neighborhood for the next...,...,Stay_12_Months:Agreement_Statements_Access_and...,GROUPED-SINGLE,@ALL,Community Capacity,OR,2023-01-01,2099-12-31,district_number,1006.0,0.659046
6,1,4,1,NaN,False,False,NaN,Please indicate how much you agree to the foll...,NaN,I plan to stay in my neighborhood for the next...,...,Stay_12_Months:Agreement_Statements_Access_and...,NaN,NaN,NaN,NaN,NaT,NaT,district_number,NaN,NaN
7,2,20,1,NaN,True,True,Agreement_Statements_Access_and_Support,Please indicate how much you agree to the foll...,Stay_12_Months,I plan to stay in my neighborhood for the next...,...,Stay_12_Months:Agreement_Statements_Access_and...,GROUPED-SINGLE,@ALL,Community Capacity,OR,2023-01-01,2099-12-31,district_number,987.0,0.020263
8,2,47,1,NaN,True,True,Agreement_Statements_Access_and_Support,Please indicate how much you agree to the foll...,Stay_12_Months,I plan to stay in my neighborhood for the next...,...,Stay_12_Months:Agreement_Statements_Access_and...,GROUPED-SINGLE,@ALL,Community Capacity,OR,2023-01-01,2099-12-31,district_number,987.0,0.047619
9,2,40,1,NaN,False,True,Agreement_Statements_Access_and_Support,Please indicate how much you agree to the foll...,Stay_12_Months,I plan to stay in my neighborhood for the next...,...,Stay_12_Months:Agreement_Statements_Access_and...,GROUPED-SINGLE,@ALL,Community Capacity,OR,2023-01-01,2099-12-31,district_number,987.0,0.040527


In [30]:
groups = datadictionary[["group", "response_type"]].drop_duplicates()

assert (
    len(groups)
    == len(groups[groups.duplicated(subset="group")])
)

AssertionError: 